# Stage 3 - Bronze Ingestion

Ingests the raw source CSV into the Bronze Delta table:
`workspace.default.capstone_bronze_sales`

- Schema is inferred from the CSV header.
- All original columns are preserved as-is (no transformations).
- A `_ingested_at` audit column (UTC timestamp) is appended.
- The `quality_flag` produced by Stage 2 is merged in from the staging table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## 1. Configuration

In [ ]:
SOURCE_PATH   = "/Volumes/workspace/default/sales_data/sales_source_1500.csv"
STAGING_TABLE = "workspace.default.capstone_staging_sales"
BRONZE_TABLE  = "workspace.default.capstone_bronze_sales"

## 2. Read staging table (with quality_flag already attached)

In [ ]:
# Prefer the pre-flagged staging table written by 00_data_quality_report.
# Fall back to raw CSV if the staging table does not yet exist.
try:
    df_staged = spark.read.table(STAGING_TABLE)
    print(f"Loaded staging table: {STAGING_TABLE} - {df_staged.count()} rows")
except Exception:
    print("Staging table not found - reading raw CSV and applying default flag.")
    df_staged = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(SOURCE_PATH)
        .withColumn("quality_flag", F.lit("VALID"))
    )

## 3. Add audit column and write to Bronze

In [ ]:
df_bronze = df_staged.withColumn("_ingested_at", F.current_timestamp())

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

count = spark.read.table(BRONZE_TABLE).count()
print(f"Bronze table written: {BRONZE_TABLE}")
print(f"Row count           : {count}")

## 4. Quick validation

In [ ]:
print("=== Bronze table schema ===")
spark.read.table(BRONZE_TABLE).printSchema()

print("=== Sample rows ===")
spark.read.table(BRONZE_TABLE).show(5, truncate=False)

print("=== quality_flag distribution ===")
(
    spark.read.table(BRONZE_TABLE)
    .groupBy("quality_flag")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)